# Layer 2 (FRIENDS) vs PARTIAL poisoning: pilot runs on Colab (CIFAR-100, EWC)

Trains the poisoned last task (task 9) from the authors' checkpoint under six conditions and prints one table:

| Arm | Data | Defense | What it tells us |
|---|---|---|---|
| A | clean | none | the target an ideal defense would recover |
| B | poisoned | none | the attack (should match your earlier stage-4 numbers) |
| F | poisoned | FRIENDS | **the main result** |
| F-clean | clean | FRIENDS | what the defense costs when there is no attack |
| F-fr | poisoned | friendly noise only (ζ = 32/255) | contribution of the friendly part |
| F-rn | poisoned | random noise only (μ = 32/255) | contribution of the random part |

(In the paper, a component that runs alone gets double the budget, so the totals match.)

**Before you start:** *Runtime > Change runtime type > GPU*. Each run takes a few minutes; all six about 30–40 minutes on a T4.
If Colab disconnects, run the notebook again: finished arms are skipped.

**What is new in this notebook.** Only a random fraction `POISON_FRAC` of the last task is poisoned (e.g. 0.2 = 20%); the rest stays clean.
Section 5b builds a *derived* noise file with `partial_poison.py` (the stage-3 noise is zeroed on the un-poisoned samples) and the six arms
then run **unchanged** on that file. No stage-3 re-run, and none of the repo / Layer-2 code is edited.

**Before you start:** the repo is used in place from Drive (no cloning); make sure `partial_poison.py` is at `PARTIAL_PY` (section 1). The sbatch file is not needed on Colab.
Note: 20% of the *last task* is 2% of all CIFAR-100 training data (10 tasks); for 5% of all data use `POISON_FRAC = 0.5`.

## 1. Settings: edit these paths

In [ ]:
NOISE_PKL = '/content/drive/MyDrive/brainwash/brainwash-repro/repro/sweep/noise/cifar100_ewc_cautious_eps0.3/noise_ewc_wcur_1.0_cifar100_ewc_cautious_eps0.3__delta_0.3_dataset_split_cifar100_target_task_0_attacked_task_9_noise_optim_lr_0.005__n_iters_1_n_epochs_5000_seed_0_mode_cautious__w_cur_1.0___min_acc_target_29.pkl'   # <- the stage-3 noise file to attack with
RESULTS_DIR = '/content/drive/MyDrive/brainwash/layer2_runs/partial_p0.2_ps0'                           # logs, accuracy matrices and summary go here
# !! Use a NEW RESULTS_DIR whenever you change NOISE_PKL, POISON_FRAC or POISON_SEED: arms that already have a finished log
# !! in RESULTS_DIR are skipped, so the old results would be reused silently.
SEED = 0

# --- partial poisoning ---
POISON_FRAC = 0.2        # fraction of the LAST TASK that is poisoned (0.2 = 20% = 1000 of 5000 images = 2% of all training data). 1.0 = full poison
POISON_SEED = 0          # which random subset is poisoned; repeat with 1, 2 for error bars
PARTIAL_PY  = '/content/drive/MyDrive/brainwash/brainwash-repro/partial_poison_colab.py'   # the wrapper file
POISON_DIR  = '/content/drive/MyDrive/brainwash/brainwash-repro/repro/noise_partial'       # derived (masked) noise files are cached here

# The repo lives on Drive and is used in place (nothing is cloned). If you ever move it, change REPO_DIR AND the
# three '%writefile' paths and the '%cd' / 'cd' lines further down (a %writefile path cannot use a variable).
REPO_DIR = '/content/drive/MyDrive/brainwash/brainwash-repro'


## 2. GPU check and Google Drive

In [ ]:
import torch, os
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > GPU'
print(torch.__version__, torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(RESULTS_DIR, exist_ok=True)
assert os.path.isfile(NOISE_PKL), f'not found: {NOISE_PKL}'
print('noise pkl:', NOISE_PKL, f'({os.path.getsize(NOISE_PKL)/1e6:.0f} MB)')assert os.path.isfile(PARTIAL_PY), f'copy partial_poison.py to Drive first: {PARTIAL_PY}'


## 3. Code

The repo is used directly from Drive (`REPO_DIR` in section 1), so nothing is cloned. The next three cells (section 4) write the Layer-2 files into `REPO_DIR/layer2_friends/` on Drive; re-running them just rewrites the same files.

In [ ]:
assert os.path.isdir(REPO_DIR), f'repo not found on Drive: {REPO_DIR}'
os.makedirs(f'{REPO_DIR}/layer2_friends', exist_ok=True)
for f in ('main_baselines.py', 'approaches/arguments.py', 'resnet.py'):
    assert os.path.isfile(f'{REPO_DIR}/{f}'), f'missing {f}: is REPO_DIR the brainwash-repro root?'
print('using repo on Drive:', REPO_DIR)

## 4. The Layer-2 files (not in git; the next three cells write them)

* `friends.py`: the friendly noise (port of the official function) and the random noise.
* `friends_training.py`: the Layer-2 training of EWC / RWALK / AFEC: copies of the originals plus marked `# >>> LAYER2` blocks.
* `friends_layer2.py`: the stage-4 script (a fork of `main_baselines.py`, same style as your Layer-1 scripts).

In [ ]:
%%writefile /content/drive/MyDrive/brainwash/brainwash-repro/layer2_friends/friends.py
"""
friends.py

FRIENDS (Liu, Yang & Mirzasoleiman, NeurIPS 2022) building blocks for the
Layer-2 defense against BrainWash. Two pieces:

  generate_friendly_noise(...)  the friendly component eta_i (computed once per task)
  RandomNoise(...)              the random component r (fresh every training step)

generate_friendly_noise is a PORT of the function with the same name in the
official repo (https://github.com/tianyu139/friendly-noise, file friendly_noise.py).
For a given model, input batch and RNG state it performs the identical
computation -- validate_friends.py --checks v1 asserts that numerically. The only
changes are generalizations the BrainWash pipeline needs:

  * no hard-coded 3x32x32 / 10-class buffers  (miniImageNet is 84x84, tiny is 64x64)
  * takes a tensor instead of a DataLoader     (stage 4 holds xtrain as one tensor)
  * multi-head model: `target` picks what must be preserved
        'head'     -> logits of head `task` (the paper's objective; loss 'KL' or 'MSE')
        'features' -> the shared 1024-d embedding before the heads   (variant (a))
  * one extra loss, 'MSE_rel' = MSE / mean(target^2): MSE made scale-free so the
    same lam (lambda) can be used on features as on logits.
  * no input normalization: the BrainWash victim consumes raw [0,1] pixels, so
    the official `transform` (Normalize) is the identity here.

Notation (thesis): eta = friendly noise (the official code and the FRIENDS paper
call it eps / epsilon; renamed because epsilon is the BrainWash poison budget
0.1 / 0.3 here), zeta = its per-pixel cap, lam = lambda, the weight of the
"make eta large" term (the official code calls it mu, which clashes with the
random-noise size mu), mu = random-noise size.

Faithfully kept from the official code (do not "fix" these without saying so in
the thesis -- they are part of what FRIENDS is):
  * objective: emp_risk - lam * mean(eta_clamped^2), SGD + Nesterov momentum 0.9
  * KL is KLDivLoss(log p(x+eta), log p(x)) = KL( p(x) || p(x+eta) ), i.e. the
    reverse direction of the paper's Eq. 3
  * init eta ~ U(-8/255, 8/255) per batch, drawn from the global torch RNG
  * the MultiStepLR scheduler is created but never stepped in the official loop,
    so the learning rate is constant; we keep it constant
  * returned noise is the unclamped iterate, clamped to [-clamp, clamp] at the end
"""

import time

import torch
import torch.nn.functional as F


# ----------------------------------------------------------------------------
# model outputs
# ----------------------------------------------------------------------------

def model_output(model, x, target, task):
    """What the friendly noise must preserve, for the BrainWash multi-head ResNet."""
    if target == 'head':
        return model(x)[task]
    if target == 'features':
        # resnet.ResNet returns the pooled embedding when include_head is False
        old = model.include_head
        model.include_head = False
        try:
            return model(x)
        finally:
            model.include_head = old
    raise ValueError(f'unknown target {target!r}')


# ----------------------------------------------------------------------------
# friendly component
# ----------------------------------------------------------------------------

def generate_friendly_noise(model, x, *, task=-1, target='head', loss_fn='KL',
                            friendly_epochs=30, lam=1.0, friendly_lr=100.,
                            friendly_momentum=0.9, nesterov=True,
                            clamp=16 / 255, init_mag_255=8,
                            batch_size=128, device='cuda', verbose=False,
                            return_trace=False):
    """Friendly noise for every image in x ([N,C,H,W] in [0,1]).

    Returns a CPU tensor shaped like x, in [-clamp, clamp]. With
    return_trace=True also returns per-batch (first, last) values of the
    empirical-risk and constraint terms, which V2 uses to check that the
    optimization actually moved.
    """
    if loss_fn == 'KL':
        criterion = torch.nn.KLDivLoss(reduction='batchmean', log_target=True)
    elif loss_fn in ('MSE', 'MSE_rel'):
        criterion = torch.nn.MSELoss()
    else:
        raise ValueError(f'unknown loss_fn {loss_fn!r}')

    was_training = model.training
    model.eval()

    n = x.shape[0]
    friendly_noise = torch.zeros(x.shape, device='cpu')
    trace = []

    for batch_idx, lo in enumerate(range(0, n, batch_size)):
        inputs = x[lo:lo + batch_size].to(device)
        # same draw as the official code: global RNG, shape of the batch
        # same arithmetic, in the same order, as the official `(torch.rand(...) - 0.5) * 2 * 8/255`:
        # a differently-rounded start (e.g. times a pre-computed 8/255) is enough to send the
        # (chaotic) optimization to a different result
        init = (torch.rand(*(inputs.shape)) - 0.5) * 2 * init_mag_255 / 255
        eta = init.to(device).requires_grad_(True)
        optimizer = torch.optim.SGD([eta], lr=friendly_lr, momentum=friendly_momentum,
                                    nesterov=nesterov)

        output_original = model_output(model, inputs, target, task)
        if loss_fn == 'KL':
            output_original = F.log_softmax(output_original, dim=1).detach()
        else:
            output_original = output_original.detach()
        scale = output_original.pow(2).mean().clamp_min(1e-12) if loss_fn == 'MSE_rel' else 1.0

        first = None
        for friendly_epoch in range(friendly_epochs):
            eta_clamp = torch.clamp(eta, -clamp, clamp)
            perturbed = torch.clamp(inputs + eta_clamp, 0, 1)
            output_perturb = model_output(model, perturbed, target, task)
            if loss_fn == 'KL':
                output_perturb = F.log_softmax(output_perturb, dim=1)

            emp_risk = criterion(output_perturb, output_original) / scale
            constraint = torch.mean(torch.square(eta_clamp))
            loss = emp_risk - lam * constraint

            optimizer.zero_grad()
            loss.backward()
            model.zero_grad()
            optimizer.step()

            if first is None:
                first = (emp_risk.item(), constraint.item())
            if verbose:
                print(f'  friendly batch {batch_idx} epoch {friendly_epoch}: '
                      f'emp_risk={emp_risk.item():.4g} constraint={constraint.item():.4g}',
                      end='\r', flush=True)

        trace.append({'first': first, 'last': (emp_risk.item(), constraint.item())})
        friendly_noise[lo:lo + batch_size] = eta.detach().cpu()

    if verbose:
        print()
    friendly_noise = torch.clamp(friendly_noise, -clamp, clamp)
    model.zero_grad(set_to_none=True)
    model.train(was_training)

    if return_trace:
        return friendly_noise, trace
    return friendly_noise


# ----------------------------------------------------------------------------
# random component
# ----------------------------------------------------------------------------

def sample_random_noise(shape, mu, kind, generator, device, dtype=torch.float32):
    """Bernoulli {-mu,+mu}, Uniform [-mu,mu] or Gaussian N(0, mu^2) -- the paper's
    three FRIENDS-B / -U / -G options."""
    if kind == 'bernoulli':
        r = torch.rand(shape, generator=generator, device=device, dtype=dtype)
        return ((r > 0.5).to(dtype) * 2 - 1) * mu
    if kind == 'uniform':
        r = torch.rand(shape, generator=generator, device=device, dtype=dtype)
        return (r * 2 - 1) * mu
    if kind == 'gaussian':
        return torch.randn(shape, generator=generator, device=device, dtype=dtype) * mu
    raise ValueError(f'unknown random noise kind {kind!r}')


class RandomNoise:
    """The random component r: fresh random noise for every training batch.

    Called explicitly, on one marked line inside the Layer-2 copy of the
    victim's training step (layer2_friends/friends_training.py):

        images = self.random_noise(images)

    Each call draws new noise (Bernoulli: every pixel gets +mu or -mu), adds it
    and clips the result to [0, 1]. It has its own random-number generator, so
    the batch order of training (numpy shuffle) is exactly the same as in the
    undefended run. mu = 0 returns the images untouched.

    calls counts how many batches were noised; with track=True, checksums holds
    the noise sum of the first few calls (V4 uses it to check draws are fresh).
    """

    def __init__(self, mu, kind='bernoulli', seed=0, clamp01=True, track=False):
        self.mu, self.kind, self.seed, self.clamp01, self.track = mu, kind, seed, clamp01, track
        self.calls = 0
        self.checksums = []
        self._gen = None
        self.enabled = True  # FriendsDefense can hold the noise back until its start epoch

    def __call__(self, images):
        if self.mu == 0 or not self.enabled:
            return images
        if self._gen is None:
            self._gen = torch.Generator(device=images.device)
            self._gen.manual_seed(self.seed)
        noise = sample_random_noise(images.shape, self.mu, self.kind, self._gen, images.device, images.dtype)
        out = images + noise
        if self.clamp01:
            out = torch.clamp(out, 0, 1)
        self.calls += 1
        if self.track and len(self.checksums) < 4:
            self.checksums.append(float(noise.double().sum()))
        return out


# ----------------------------------------------------------------------------
# the defense: FRIENDS as the paper's Algorithm 2, for one continual-learning task
# ----------------------------------------------------------------------------

class FriendsDefense:
    """FRIENDS settings for one task, and the one step that needs the model.

    Paper's Algorithm 2 on the task being learned (the poisoned last task):
      * epochs 0 .. warmup_epochs-1 : train on x, plus fresh random noise r every step
      * at epoch == warmup_epochs   : compute the friendly noise eta ONCE, from the
                                      CURRENT model (head T, KL), for every training
                                      image; from here on the training images are x + eta
      * remaining epochs            : train on x + eta, plus fresh random noise r

    The three Layer-2 trainers (friends_training.py) call on_epoch_start() from one marked
    block at the top of their epoch loop; attach() gives them r. random_start_epoch > 0 holds
    r back until that epoch (e.g. = warmup_epochs: a plain warm-up, then eta and r together).

    use_friendly / use_random switch the two components off for the ablation
    ("friendly only" / "random only"; the paper matches the budgets: zeta=32 for
    friendly only, mu=32 for random only -- set them on the command line).

    log is filled when eta is computed (printed, and written to summary_layer2.tsv).
    """

    def __init__(self, zeta255=16, mu255=16, kind='bernoulli', warmup_epochs=2, friendly_epochs=30,
                 friendly_lr=100., friendly_lambda=1., friendly_batch=128, use_friendly=True,
                 use_random=True, random_start_epoch=0, seed=0, device='cuda'):
        self.zeta, self.mu = zeta255 / 255, mu255 / 255
        self.kind, self.warmup_epochs = kind, warmup_epochs
        self.friendly_epochs, self.friendly_lr, self.friendly_lambda = friendly_epochs, friendly_lr, friendly_lambda
        self.friendly_batch, self.use_friendly, self.use_random = friendly_batch, use_friendly, use_random
        self.random_start_epoch = random_start_epoch  # 0 = from the first epoch (the paper)
        self.seed, self.device = seed, device
        self.log = {'calls': 0}

    def attach(self, appr):
        """Hand r (and this object) to a Layer-2 learner."""
        appr.friends = self
        if self.use_random:
            appr.random_noise = RandomNoise(self.mu, self.kind, seed=self.seed)
            if hasattr(appr, 'random_noise_emp'):                  # AFEC's second network
                appr.random_noise_emp = RandomNoise(self.mu, self.kind, seed=self.seed + 1)
            if self.random_start_epoch > 0:                        # held back until its start epoch
                appr.random_noise.enabled = False
                if hasattr(appr, 'random_noise_emp'):
                    appr.random_noise_emp.enabled = False

    def on_epoch_start(self, appr, xtrain, t, data, epoch):
        """Called at the top of every epoch by the Layer-2 train(): switches the random noise on at its
        start epoch (if it was held back) and computes the friendly noise once, after the warm-up."""
        if self.use_random and self.random_start_epoch > 0 and epoch == self.random_start_epoch:
            appr.random_noise.enabled = True
            if hasattr(appr, 'random_noise_emp'):
                appr.random_noise_emp.enabled = True
            print(f'[LAYER2] random noise switched on at epoch {epoch}')
        if self.use_friendly and epoch == self.warmup_epochs:
            xtrain = self.add_friendly_noise(appr, xtrain, t, data, epoch)
        return xtrain

    def add_friendly_noise(self, appr, xtrain, t, data, epoch):
        """eta from the model as it is NOW; returns clamp(xtrain + eta, 0, 1)."""
        t0 = time.time()
        old = []
        for u in range(t):  # how much of the old tasks is already gone when eta is computed
            xt, yt = data[u]['test']['x'].cuda(), data[u]['test']['y'].cuda()
            old.append(100 * float(appr.eval(u, xt, yt)[1]))
        eta = generate_friendly_noise(appr.model, xtrain, task=t, target='head', loss_fn='KL',
                                      friendly_epochs=self.friendly_epochs, lam=self.friendly_lambda,
                                      friendly_lr=self.friendly_lr, clamp=self.zeta,
                                      batch_size=self.friendly_batch, device=self.device)
        x_new = torch.clamp(xtrain + eta.to(xtrain.device), 0, 1)
        a = eta.abs()
        self.log = {'calls': self.log['calls'] + 1, 'epoch': epoch, 'seconds': time.time() - t0,
                    'old_task_acc_at_eta': float(sum(old) / len(old)) if old else float('nan'),
                    'eta_mean_over_zeta': a.mean().item() / self.zeta,
                    'eta_frac_at_bound': (a >= 0.99 * self.zeta).float().mean().item(),
                    'linf_over_zeta': a.max().item() / self.zeta,
                    'input_change_linf': (x_new - xtrain).abs().max().item()}
        L = self.log
        print(f"[LAYER2] friendly noise computed at epoch {epoch} (after {epoch} warm-up epoch(s)): "
              f"mean|eta|/zeta={L['eta_mean_over_zeta']:.3f}  at-bound={100 * L['eta_frac_at_bound']:.1f}%  "
              f"Linf/zeta={L['linf_over_zeta']:.3f}  old-task acc at that moment={L['old_task_acc_at_eta']:.1f}%  "
              f"[{L['seconds']:.0f}s]")
        return x_new

In [ ]:
%%writefile /content/drive/MyDrive/brainwash/brainwash-repro/layer2_friends/friends_training.py
"""
friends_training.py

Layer-2 (FRIENDS) versions of the victim's training for EWC, RWALK and AFEC.

Each class below inherits everything from the original approaches/<method>.py
learner and replaces only the functions that run the training of a task:

  train(...)        copy of the original, plus ONE marked block at the top of the
                    epoch loop:  at epoch == warm-up length, compute the friendly
                    noise eta from the current model and train on x + eta from then on
                    (and switch the random noise on at its start epoch, if held back)
  train_epoch(...)  copy of the original, plus ONE marked line after a batch of
                    images is taken: fresh random noise r on every step
                    (AFEC also train_emp_epoch for its second network)

The added lines are the ones between the markers:

    # >>> LAYER2: ...
    # <<< LAYER2

Everything else is the original code (only commented-out dead code and unused
timing variables were left out). approaches/*.py and the attacker's code
(main_brainwash.py) are not touched, evaluation is untouched, and with no
FriendsDefense attached these trainers behave exactly like the originals --
validate_friends.py V4 checks that by training both and comparing the weights.

The friendly noise is computed from the model after the warm-up epochs, so the
model has already trained on `warmup_epochs` epochs of the (poisoned) data when
eta is computed -- that is the paper's Algorithm 2 (5 warm-up epochs of 80).

NOTE: importing this module imports approaches/*, which parse sys.argv at
import time (they call get_args()); set sys.argv first, as main_baselines.py
style scripts do.
"""

import time
from copy import deepcopy

import numpy as np
import torch

from approaches import ewc, rwalk, afec_ewc


def _no_noise(images):
    return images


class EWC(ewc.Appr):
    random_noise = staticmethod(_no_noise)
    friends = None  # a FriendsDefense (friends.py), attached by FriendsDefense.attach()

    def train(self, t, xtrain, ytrain, xvalid, yvalid, data, input_size, taskcla):
        best_loss = np.inf
        best_model = ewc.utils.get_model(self.model)
        lr = self.lr
        self.optimizer = self._get_optimizer(lr)

        # Loop epochs
        for e in range(self.nepochs):
            # >>> LAYER2: FRIENDS: friendly noise eta once after the warm-up epochs; random noise from its start epoch
            if self.friends is not None:
                xtrain = self.friends.on_epoch_start(self, xtrain, t, data, e)
            # <<< LAYER2

            # Train
            self.train_epoch(t, xtrain, ytrain, e)

            # Valid
            valid_loss, valid_acc = self.eval(t, xvalid, yvalid)
            print('task: {} epoch {} Valid: loss={:.3f}, acc={:5.1f}% |'.format(t, e, valid_loss, 100 * valid_acc), end='')
            print(' lr : {:.6f}'.format(self.optimizer.param_groups[0]['lr']))
            print()
            best_model = ewc.utils.get_model(self.model)

        # Restore best
        ewc.utils.set_model_(self.model, best_model)

        # Update old
        self.model_old = deepcopy(self.model)
        self.model_old.train()
        ewc.utils.freeze_model(self.model_old) # Freeze the weights

        # Fisher ops
        if t > 0:
            fisher_old = {}
            for n, _ in self.model.named_parameters():
                fisher_old[n] = self.fisher[n].clone()
        self.fisher = ewc.utils.fisher_matrix_diag(t, xtrain, ytrain, self.model, self.criterion)
        if t > 0:
            # Watch out! We do not want to keep t models (or fisher diagonals) in memory, therefore we have to merge fisher diagonals
            for n, _ in self.model.named_parameters():
                self.fisher[n] = (self.fisher[n] + fisher_old[n] * t) / (t + 1)       # Checked: it is better than the other option

        for p_idx, (n, p) in enumerate(self.model.named_parameters()):
            self.model.register_buffer('{}_fisher'.format(n.replace('.', '_')), self.fisher[n])

        return


    def train_epoch(self, t, x, y, epoch):
        self.model.train()

        r = np.arange(x.size(0))
        np.random.shuffle(r)
        r = torch.LongTensor(r)

        # Loop batches
        for i in range(0, len(r), self.sbatch):
            if i + self.sbatch <= len(r): b = r[i:i + self.sbatch]
            else: b = r[i:]
            images = x[b].to(ewc.device)
            targets = y[b].to(ewc.device)

            # >>> LAYER2: FRIENDS random noise r, fresh every training step
            images = self.random_noise(images)
            # <<< LAYER2

            # Forward current model
            outputs = self.model.forward(images)[t]
            loss = self.criterion(t, outputs, targets)

            # Backward
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.clipgrad)
            self.optimizer.step()
        return


class RWALK(rwalk.Appr):
    random_noise = staticmethod(_no_noise)
    friends = None  # a FriendsDefense (friends.py), attached by FriendsDefense.attach()

    def train(self, t, xtrain, ytrain, xvalid, yvalid, data, input_size, taskcla):
        best_loss = np.inf
        best_model = rwalk.utils.get_model(self.model)
        lr = self.lr
        patience = self.lr_patience
        self.optimizer = self._get_optimizer(lr)

        # Loop epochs
        for e in range(self.nepochs):
            # >>> LAYER2: FRIENDS: friendly noise eta once after the warm-up epochs; random noise from its start epoch
            if self.friends is not None:
                xtrain = self.friends.on_epoch_start(self, xtrain, t, data, e)
            # <<< LAYER2

            # Train
            self.train_epoch(t, xtrain, ytrain)

            # Valid
            valid_loss, valid_acc = self.eval(t, xvalid, yvalid)
            print(' Valid: loss={:.3f}, acc={:5.1f}% |'.format(valid_loss, 100 * valid_acc), end='')
            print()

            best_model = rwalk.utils.get_model(self.model)
            print()
        # Restore best
        rwalk.utils.set_model_(self.model, best_model)

        # Update old
        self.model_old = deepcopy(self.model)
        rwalk.utils.freeze_model(self.model_old) # Freeze the weights

        # Update fisher & s
        for n, p in self.model.named_parameters():
            if p.requires_grad:
                if p.grad is not None:
                    self.fisher[n] = self.fisher_running[n].clone()
                    self.s[n] = (1/2) * self.s_running[n].clone()
                    self.s_running[n] = self.s[n].clone()

        self.register_fisher_into_buffer()
        self.rigister_s_into_buffer()

        return


    def train_epoch(self, t, x, y):
        self.model.train()

        r = np.arange(x.size(0))
        np.random.shuffle(r)
        r = torch.LongTensor(r)

        # Loop batches
        for i in range(0, len(r), self.sbatch):
            if i + self.sbatch <= len(r): b = r[i:i + self.sbatch]
            else: b = r[i:]
            images = x[b]
            targets = y[b]

            images, targets = images.cuda(), targets.cuda()

            # >>> LAYER2: FRIENDS random noise r, fresh every training step
            images = self.random_noise(images)
            # <<< LAYER2

            # Forward current model
            outputs = self.model.forward(images)[t]
            loss = self.criterion(t, outputs, targets)

            # Backward
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.clipgrad)
            self.optimizer.step()

            # Compute Fisher & s
            self.update_fisher_and_s()

        return


class AFEC(afec_ewc.Appr):
    random_noise = staticmethod(_no_noise)      # main network
    random_noise_emp = staticmethod(_no_noise)  # AFEC's second ("empty") network
    friends = None  # a FriendsDefense (friends.py), attached by FriendsDefense.attach()

    def train(self, t, xtrain, ytrain, xvalid, yvalid, data, input_size, taskcla):
        best_loss = np.inf
        best_model = afec_ewc.utils.get_model(self.model)
        lr = self.lr
        self.optimizer = self._get_optimizer(lr)
        self.optimizer_emp = self._get_optimizer_emp(lr)
        self.add_emp = 0

        if t == 0:
            self.model_emp = deepcopy(self.model) #use the same initialization
            self.model_emp_tmp = deepcopy(self.model)

        # Loop epochs
        for e in range(self.nepochs):
            # >>> LAYER2: FRIENDS: friendly noise eta once after the warm-up epochs; random noise from its start epoch
            if self.friends is not None:
                xtrain = self.friends.on_epoch_start(self, xtrain, t, data, e)
            # <<< LAYER2

            # Train
            clock0 = time.time()

            num_batch = xtrain.size(0)

            #train the empty net and measure fim
            if t > self.add_emp-1:

                self.train_emp_epoch(t, xtrain, ytrain, e)
                # freeze the empty net
                self.model_emp_tmp = deepcopy(self.model_emp)
                self.model_emp_tmp.train()
                afec_ewc.utils.freeze_model(self.model_emp_tmp)

                # Fisher ops
                self.fisher_emp, _ = afec_ewc.utils.fisher_matrix_diag_emp(t, xtrain, ytrain, self.model_emp, self.criterion)

            self.train_epoch(t, xtrain, ytrain, e)

            clock1 = time.time()
            train_loss, train_acc = self.eval(t, xtrain, ytrain)
            clock2 = time.time()
            print('| Epoch {:3d}, time={:5.1f}ms/{:5.1f}ms | Train: loss={:.3f}, acc={:5.1f}% |'.format(
                e+1, 1000*self.sbatch*(clock1-clock0)/num_batch,
                1000*self.sbatch*(clock2-clock1)/num_batch, train_loss, 100*train_acc), end='')
            # Valid
            valid_loss, valid_acc = self.eval(t, xvalid, yvalid)
            print(' Valid: loss={:.3f}, acc={:5.1f}% |'.format(valid_loss, 100*valid_acc), end='')
            print(' lr : {:.6f}'.format(self.optimizer.param_groups[0]['lr']))

            best_model = afec_ewc.utils.get_model(self.model)

            # after pretrain in task 0, copy the PT model as empty
            if t == 0:
                self.model_pt = deepcopy(self.model)

        # Restore best
        afec_ewc.utils.set_model_(self.model, best_model)

        # Update old
        self.model_old = deepcopy(self.model)
        self.model_old.train()
        afec_ewc.utils.freeze_model(self.model_old) # Freeze the weights

        # Fisher ops
        if t > 0:
            fisher_old = {}
            for n, _ in self.model.named_parameters():
                if 'heads' not in n:
                    fisher_old[n] = self.fisher[n].clone()
        self.fisher = afec_ewc.utils.fisher_matrix_diag(t, xtrain, ytrain, self.model, self.criterion)
        if t > 0:
            # Watch out! We do not want to keep t models (or fisher diagonals) in memory, therefore we have to merge fisher diagonals
            for n, _ in self.model.named_parameters():
                if 'heads' not in n:
                    self.fisher[n] = (self.fisher[n] + fisher_old[n] * t) / (t + 1)       # Checked: it is better than the other option

        self.register_fisher_into_buffer()

        return


    def train_epoch(self, t, x, y, epoch):
        self.model.train()

        r = np.arange(x.size(0))
        np.random.shuffle(r)
        r = torch.LongTensor(r)

        # Loop batches
        for i in range(0, len(r), self.sbatch):
            if i + self.sbatch <= len(r): b = r[i:i + self.sbatch]
            else: b = r[i:]
            images = x[b].to(afec_ewc.device)
            targets = y[b].to(afec_ewc.device)

            # >>> LAYER2: FRIENDS random noise r, fresh every training step
            images = self.random_noise(images)
            # <<< LAYER2

            # Forward current model
            outputs = self.model.forward(images)[t]
            loss = self.ce(outputs, targets)

            if t > self.add_emp:
                loss_fg = self.criterion_fg(t)
                loss += loss_fg

            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.clipgrad)
            self.optimizer.step()

            del loss
            del images, targets, outputs

        return

    def train_emp_epoch(self, t, x, y, epoch):
        self.model_emp.train()

        r = np.arange(x.size(0))
        np.random.shuffle(r)
        r = torch.LongTensor(r)

        # Loop batches
        for i in range(0, len(r), self.sbatch):
            if i + self.sbatch <= len(r): b = r[i:i + self.sbatch]
            else: b = r[i:]
            images = x[b].to(afec_ewc.device)
            targets = y[b].to(afec_ewc.device)

            # >>> LAYER2: FRIENDS random noise r, fresh every training step
            images = self.random_noise_emp(images)
            # <<< LAYER2

            # train empty net
            # Forward current model
            outputs = self.model_emp.forward(images)[t]
            loss = self.ce(outputs, targets)

            # Backward
            self.optimizer_emp.zero_grad()
            loss.backward()
            self.optimizer_emp.step()

        return


TRAINERS = {'ewc': EWC, 'rwalk': RWALK, 'afec_ewc': AFEC}

In [ ]:
%%writefile /content/drive/MyDrive/brainwash/brainwash-repro/layer2_friends/friends_layer2.py
"""
friends_layer2.py

Layer 2 defense against BrainWash: FRIENDS (Liu, Yang & Mirzasoleiman, NeurIPS 2022),
applied while the victim continual learner trains on the poisoned last task.

This is a deliberate fork of main_baselines.py (stage 4), NOT an edit of it, in the
same style as the Layer-1 scripts: the original reproduction stays byte-for-byte
reproducible, and every line that is not part of the original main_baselines.py is
marked "# >>> LAYER2" / "# <<< LAYER2". Without --friends this script does exactly
what main_baselines.py does (only the output location differs).

What --friends does (paper's Algorithm 2, on the poisoned task only):
  * warm-up epochs : train on the poisoned images, with fresh random noise r each step
  * then           : compute the friendly noise eta ONCE from the current model
                     (head of the task, KL divergence), for every training image
  * remaining      : train on image + eta, with fresh random noise r each step
The training code that does this is in friends_training.py (copies of each method's
train()/train_epoch() plus marked lines); the attacker's code and approaches/*.py are
not touched.

Arms, selected by flags (same flags as main_baselines.py for the data):
    clean                        --checkpoint <noise.pkl>
    poison                       --checkpoint <noise.pkl> --addnoise
    poison + FRIENDS             --checkpoint <noise.pkl> --addnoise --friends
    clean  + FRIENDS (cost)      --checkpoint <noise.pkl> --friends
    friendly noise only          ... --friends --components friendly_only --zeta 32
    random noise only            ... --friends --components random_only  --mu_rand 32
(the paper matches the budgets in the component ablation: 32 when a component runs alone)

Notation: eta = friendly noise, zeta = its per-pixel cap, r = random noise, mu = its
size (+-mu per pixel for Bernoulli), lambda = weight of the "make eta large" term.
epsilon is the BrainWash poison budget (0.1 / 0.3), as everywhere else in the thesis.

Results: acc_mat_*.npy and summary_layer2.tsv in --out_dir
(default: $LAYER2_OUT_DIR, else layer2_friends/results/runs).
"""
import sys, os, time
import numpy as np
import pickle as pkl

# >>> LAYER2: this file lives in layer2_friends/; the repo root (utils, resnet, approaches) and
# this folder (friends, friends_training) must be importable.
HERE = os.path.dirname(os.path.abspath(__file__))
REPO_ROOT = os.path.dirname(HERE)
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, HERE)
# <<< LAYER2
import utils
import torch
from approaches.arguments import get_args
from resnet import ResNet18


tstart = time.time()

def main(args):

    if args.checkpoint != None:
        checkpoint_dict = pkl.load(open(args.checkpoint, 'rb'))   

    if args.approach == 'afec_ewc' or args.approach == 'ewc' or args.approach == 'afec_rwalk' or args.approach == 'rwalk' or args.approach == 'afec_mas' or args.approach == 'mas' or args.approach == 'afec_si' or args.approach == 'si' or args.approach == 'ft' or args.approach == 'random_init' or args.approach == 'rwalk2':
        log_name = '{}_{}_{}_{}_lamb_{}_lr_{}_batch_{}_epoch_{}'.format(args.date, args.experiment, args.approach,args.seed,
                                                                        args.lamb, args.lr, args.batch_size, args.nepochs)
    elif args.approach == 'gs':
        log_name = '{}_{}_{}_{}_lamb_{}_mu_{}_rho_{}_eta_{}_lr_{}_batch_{}_epoch_{}'.format(args.date, args.experiment,
                                                                                            args.approach, args.seed, 
                                                                                            args.lamb, args.mu, args.rho,
                                                                                                    args.eta,
                                                                                            args.lr, args.batch_size, args.nepochs)

    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(args.seed)
    else:
        print('[CUDA unavailable]'); sys.exit()
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Args -- Experiment
    if args.experiment == 'split_cifar100':
        # from dataloaders import split_cifar100 as dataloader
        from approaches.data_utils import generate_split_cifar100_tasks
    elif args.experiment == 'split_mini_imagenet':
        from approaches.data_utils import generate_split_mini_imagenet_tasks
    elif args.experiment == 'split_tiny_imagenet':
        from approaches.data_utils import generate_split_tiny_imagenet_tasks


    # Args -- Approach

    if args.approach == 'afec_ewc':
        from approaches import afec_ewc as approach
    elif args.approach == 'ewc':
        from approaches import ewc as approach
    elif args.approach == 'rwalk':
        from approaches import rwalk as approach
    elif args.approach == 'mas':
        from approaches import mas as approach


    print('Load data...')

    if args.experiment == 'split_cifar100':
        order = np.arange(100)
        im_sz = 32
        emb_fact = 1    
        data, taskcla, inputsize, task_order = generate_split_cifar100_tasks(args.tasknum, args.seed, rnd_order=False, order=order)
    elif args.experiment == 'split_mini_imagenet':
        order = np.arange(100)  
            
        class_num = 100 // (args.tasknum)  
        im_sz = 84
        emb_fact = 1    

        order = np.arange(100)
        home = os.path.expanduser('~')
        mini_root = os.path.join(home, 'data', 'miniImagenet' ) 
        data, taskcla, inputsize, task_order = generate_split_mini_imagenet_tasks(mini_root, task_num = args.tasknum, 
                                                                    rnd_order=False, order=order) 
        
    elif args.experiment == 'split_tiny_imagenet':
        order = np.arange(200)  
        home = os.path.expanduser('~')  
        root_add = os.path.join(home, 'data', 'tiny-imagenet-200') 
        dataset_file = './data/tiny_imagenet.npz'
        data, taskcla, inputsize, task_order = generate_split_tiny_imagenet_tasks(task_num = args.tasknum, 
                                                                    rnd_order=False, save_data=False,
                                                                    dataset_file=dataset_file, 
                                                                    order=order, root_add=root_add)
        
        class_num = 200 // (args.tasknum)  
        im_sz = 64
        emb_fact = 9
        
    print('\nInput size =', inputsize, '\nTask info =', taskcla)


    ########################################################################################################################

    print('Inits...')
    torch.set_default_tensor_type('torch.cuda.FloatTensor')

    nf = 32

    net = ResNet18(args.tasknum, data['ncla']//args.tasknum, nf=nf, include_head=True).cuda()
    net_emp = ResNet18(args.tasknum, data['ncla']//args.tasknum, nf=nf, include_head=True).cuda()


    ########################################################################################################################

    save_dict = {}  
    save_dict['scenario'] = args.scenario_name
    save_dict['model_type'] = 'resnet'    
    save_dict['dataset'] = args.experiment
    save_dict['class_num'] = data['ncla'] // args.tasknum  
    save_dict['bs'] = args.batch_size
    save_dict['lr'] = args.lr
    save_dict['n_epochs'] = args.nepochs
    save_dict['model'] = net.state_dict()
    save_dict['model_name'] = net.__class__.__name__
    save_dict['task_num'] = args.lasttask        
    save_dict['task_order'] = task_order
    save_dict['seed'] = args.seed    
    save_dict['emb_fact'] = emb_fact  
    save_dict['im_sz'] = inputsize[1]  

    cont_method_args = {'method': args.approach} 
    for tmp_key in args.__dict__.keys():    
        cont_method_args[tmp_key] = args.__dict__[tmp_key] 

    save_dict['cont_method_args'] = cont_method_args    

    # >>> LAYER2: with --friends, the Layer-2 version of the learner (friends_training.py)
    if args.friends:
        from friends_training import TRAINERS
        Appr = TRAINERS[args.approach]
    else:
        Appr = approach.Appr
    # <<< LAYER2

    if 'afec' in args.approach:
        if args.checkpoint is not None:
            lamb = checkpoint_dict['pretrained_ckpt']['cont_method_args']['lamb']  
            lamb_emp = checkpoint_dict['pretrained_ckpt']['cont_method_args']['lamb_emp']  
        else:
            lamb = args.lamb    
            lamb_emp = args.lamb_emp    

        appr = Appr(net, sbatch=args.batch_size, lamb=lamb, lamb_emp=lamb_emp, 
                            lr=args.lr, nepochs=args.nepochs, args=args, log_name=log_name,
                            empty_net = net_emp, clipgrad=args.clip)
    else:
        if args.checkpoint is not None:
            lamb = checkpoint_dict['pretrained_ckpt']['cont_method_args']['lamb']  
        else:
            lamb = args.lamb    

        appr = Appr(net, lamb=lamb, sbatch=args.batch_size, lr=args.lr, nepochs=args.nepochs, args=args, log_name=log_name, clipgrad=args.clip)


    if args.checkpoint is not None:
        appr.load_model(checkpoint_dict['pretrained_ckpt']['model'])   
        if 'afec' in args.approach:
            appr.load_emp_model(checkpoint_dict['pretrained_ckpt']['cont_method_args']['model_emp'])

        if args.init_acc:
            accs_tmp = []
            for u in range(checkpoint_dict['pretrained_ckpt']['task_num']):  
                xtest = data[u]['test']['x']
                ytest = data[u]['test']['y']
                test_loss, test_acc = appr.eval(u, xtest, ytest)
                accs_tmp.append(test_acc *100)   
            
            with np.printoptions(precision=2, suppress=True):   
                print(np.array(accs_tmp) ) 
            
        

    # >>> LAYER2: record the hardware and software (the same code gave different early-training results in different
    # Colab sessions); then the FRIENDS settings; attach() hands the random noise to the learner
    print(f'[LAYER2] torch {torch.__version__}, cuda {torch.version.cuda}, device {torch.cuda.get_device_name(0)}')
    defense = None
    if args.friends:
        from friends import FriendsDefense
        defense = FriendsDefense(zeta255=args.zeta, mu255=args.mu_rand, kind=args.noise_kind,
                                 warmup_epochs=args.warmup_epochs, friendly_epochs=args.friendly_epochs,
                                 friendly_lr=args.friendly_lr, friendly_lambda=args.friendly_lambda,
                                 friendly_batch=args.friendly_batch,
                                 use_friendly=args.components in ('both', 'friendly_only'),
                                 use_random=args.components in ('both', 'random_only'),
                                 random_start_epoch=args.warmup_epochs if args.random_after_warmup else 0,
                                 seed=args.seed, device='cuda')
        defense.attach(appr)
        print(f'[LAYER2] FRIENDS on: components={args.components} zeta={args.zeta}/255 mu={args.mu_rand}/255 '
              f'({args.noise_kind}) warm-up={args.warmup_epochs} epoch(s) of {args.nepochs}; '
              f'random noise from epoch {args.warmup_epochs if args.random_after_warmup else 0}; '
              f'friendly noise: {args.friendly_epochs} steps, lr={args.friendly_lr}, lambda={args.friendly_lambda}')
    # <<< LAYER2

    print(appr.criterion)
    print('-' * 100)
    relevance_set = {}

    acc = np.zeros((len(taskcla), len(taskcla)), dtype=np.float32)
    lss = np.zeros((len(taskcla), len(taskcla)), dtype=np.float32)

    for t, ncla in taskcla:
        if args.checkpoint is not None and t < args.lasttask:
            print('Skip task {:2d} : {:15s}'.format(t, data[t]['name']))
            continue


        if t==1 and 'find_mu' in args.date:
            break

        if t == args.lasttask and args.checkpoint is None:  
            break
        
        print('*' * 100)
        print('Task {:2d} ({:s})'.format(t, data[t]['name']))
        print('*' * 100)

        # Get data
        xtrain = data[t]['train']['x'].clone()
        xvalid = data[t]['test']['x'].clone()   

        ytrain = data[t]['train']['y'].clone()
        yvalid = data[t]['test']['y'].clone()

        if args.checkpoint is not None and args.addnoise == True:
            
            if args.uniform is True:
                print('Using uniform noise')
                if 'inj_data_idx' not in checkpoint_dict.keys():
                    all_noise = torch.rand_like(xtrain) * 2 * checkpoint_dict['delta'] - checkpoint_dict['delta']
                else:
                    print('Using uniform noise only on the injected data')
                    noise_prm = checkpoint_dict['rnd_idx_train']   
                    xtrain = xtrain[noise_prm]  
                    ytrain = ytrain[noise_prm]
                    all_noise = torch.zeros_like(xtrain)    
                    inj_idx = checkpoint_dict['inj_data_idx']   
                    print(f'number of noisy data : {len(inj_idx)}')
                    all_noise[inj_idx] = torch.rand_like(xtrain[inj_idx]) * 2 * checkpoint_dict['delta'] - checkpoint_dict['delta']

                xtrain = torch.clamp(xtrain + all_noise, 0, 1)  

                
            else:
                print('Using noise from checkpoint')
                all_noise = checkpoint_dict['latest_noise']   
                noise_prm = checkpoint_dict['rnd_idx_train']   
                xtrain = xtrain[noise_prm]
                ytrain = ytrain[noise_prm]

            
                xtrain = torch.clamp(xtrain + all_noise, 0, 1)  
        
        task = t

        # Train
        appr.train(task, xtrain, ytrain, xvalid, yvalid, data, inputsize, taskcla)
        print('-' * 100)

        # Test
        for u in range(t + 1):
            xtest = data[u]['test']['x'].cuda()
            ytest = data[u]['test']['y'].cuda()
            test_loss, test_acc = appr.eval(u, xtest, ytest)
            print('>>> Test on task {:2d} - {:15s}: loss={:.3f}, acc={:5.1f}% <<<'.format(u, data[u]['name'], test_loss,
                                                                                        100 * test_acc))
            acc[t, u] = test_acc
            lss[t, u] = test_loss
            
        # Save
        
        print('Average accuracy={:5.1f}%'.format(100 * np.mean(acc[t,:t+1])))
        print('Save at ' + args.output)

        with np.printoptions(precision=2, suppress=True):   
            print(acc)

    if args.checkpoint is not None: 
        acc[:args.lasttask, :args.lasttask] = checkpoint_dict['pretrained_ckpt']['acc_mat'][:args.lasttask, :args.lasttask]
        
    # Done
    print('*' * 100)
    print('Accuracies =')
    for i in range(acc.shape[0]):
        print('\t', end='')
        for j in range(acc.shape[1]):
            print('{:5.1f}% '.format(100 * acc[i, j]), end='')
        print()
    print('*' * 100)
    print('Done!')

    print('[Elapsed time = {:.1f} h]'.format((time.time() - tstart) / (60 * 60)))

    if args.checkpoint is not None: 
        acc_mat_sace_name = args.checkpoint.split('/')[-1]
        #remove the .pkl extension
        acc_mat_sace_name = acc_mat_sace_name[:-4]

        if args.addnoise is False:
            method = 'clean'
        if args.addnoise and args.uniform:  
            method = 'uniform'
        elif args.addnoise and args.uniform is False:
            method = 'ours'    

        # >>> LAYER2: keep FRIENDS outputs apart from the baseline's acc_mat_*.npy
        if args.friends:
            method = f'{method}_friends-{args.components}_z{args.zeta:g}_mu{args.mu_rand:g}_w{args.warmup_epochs}'
            if args.random_after_warmup:
                method += '_ra'
        os.makedirs(args.out_dir, exist_ok=True)
        # the stage-3 noise file names are ~170 characters; file systems (and Google Drive) allow 255 per
        # name, so a long name is cut and tagged with a short hash (the full name stays in summary_layer2.tsv)
        import hashlib
        short_name = acc_mat_sace_name
        if len(short_name) > 60:
            short_name = short_name[:50] + '_' + hashlib.md5(short_name.encode()).hexdigest()[:8]
        out_npy = os.path.join(args.out_dir, f'acc_mat_{short_name}_{method}_s{args.seed}.npy')
        np.save(out_npy, acc)
        print(f'[LAYER2] saved accuracy matrix -> {out_npy}')
        # <<< LAYER2



    bwt_before = np.mean((acc[args.lasttask-1] - np.diag(acc))[:args.lasttask-1][:-1])
    avg_acc_before  = np.mean(acc[args.lasttask-1, :args.lasttask])

    if args.checkpoint is not None: 
        bwt_after = np.mean((acc[-1] - np.diag(acc))[:-1])
        avg_acc_after  = np.mean(acc[-1][:-1])
        last_task_acc = acc[-1, -1] 
        print(f'After BWT : {bwt_after} After avg acc : {avg_acc_after} Last task acc : {last_task_acc}')

        # >>> LAYER2: one-line machine-readable summary next to the .npy
        summary = os.path.join(args.out_dir, 'summary_layer2.tsv')
        new = not os.path.isfile(summary)
        L = defense.log if defense is not None else {}
        fmt = lambda k, f: (f % L[k]) if k in L else '-'
        with open(summary, 'a') as fh:
            if new:
                fh.write('method\tapproach\texperiment\tseed\tfriends\tcomponents\tzeta255\tmu255\twarmup_epochs\t'
                         'bwt_after\tavg_acc_after\tlast_task_acc\told_acc_at_eta\teta_mean_over_zeta\teta_frac_at_bound\t'
                         'noise_pkl\n')
            fh.write(f'{method}\t{args.approach}\t{args.experiment}\t{args.seed}\t{int(args.friends)}\t'
                     f'{args.components if args.friends else "-"}\t{args.zeta if args.friends else "-"}\t'
                     f'{args.mu_rand if args.friends else "-"}\t{args.warmup_epochs if args.friends else "-"}\t'
                     f'{bwt_after:.4f}\t{avg_acc_after:.4f}\t{last_task_acc:.4f}\t'
                     f'{fmt("old_task_acc_at_eta", "%.2f")}\t{fmt("eta_mean_over_zeta", "%.4f")}\t'
                     f'{fmt("eta_frac_at_bound", "%.4f")}\t{os.path.basename(args.checkpoint)}\n')
        print(f'[LAYER2] appended summary row -> {summary}')
        # <<< LAYER2

        
    print(f'Before BWT : {bwt_before} Before avg acc : {avg_acc_before}')  

    save_dict['last_task'] = int(args.lasttask)
    save_dict['acc_mat'] = acc
    save_dict['avg_acc'] = np.mean(acc[-1, :args.lasttask])
    save_dict['bwt'] = bwt_before
    save_dict['model'] = net.state_dict()
    if 'afec' in args.approach: 
        save_dict['cont_method_args']['model_emp'] = net_emp.state_dict()   

    # save_dict['optim'] = optim.state_dict()

    save_name = utils.generate_save_name(save_dict)
    if args.checkpoint is None:
        #check if the file exists and add a number to the end if it does    
        if os.path.exists(f'{args.approach}_{save_name}.pkl'):
            print(f'File {args.approach}_{save_name}.pkl already exists. Saving with a different name.')
            if 'afec' not in args.approach:
                pkl.dump(save_dict, open(f'{args.approach}_lamb_{args.lamb}_{save_name}_1.pkl', 'wb'))
            else:
                pkl.dump(save_dict, open(f'{args.approach}_lamb_{args.lamb}_lambemp_{args.lamb_emp}_{save_name}_1.pkl', 'wb'))

        else:
            if 'afec' not in args.approach:
                pkl.dump(save_dict, open(f'{args.approach}_lamb_{args.lamb}_{save_name}.pkl', 'wb'))
            else:
                pkl.dump(save_dict, open(f'{args.approach}_lamb_{args.lamb}_lambemp_{args.lamb_emp}_{save_name}.pkl', 'wb'))


if __name__ == '__main__':
    # >>> LAYER2: parse the defense flags ourselves so approaches/arguments.py never has to be
    # touched; they are removed from sys.argv before the original get_args() sees it.
    import argparse
    _p = argparse.ArgumentParser(add_help=False, allow_abbrev=False)
    _p.add_argument('--friends', action='store_true', help='turn the FRIENDS defense on')
    _p.add_argument('--components', default='both', choices=['both', 'friendly_only', 'random_only'],
                    help='both = FRIENDS; friendly_only / random_only = component ablation')
    _p.add_argument('--zeta', type=float, default=16, help='cap of the friendly noise per pixel, x/255 (paper: 16)')
    _p.add_argument('--mu_rand', type=float, default=16, help='size of the random noise, x/255 (paper: 16)')
    _p.add_argument('--noise_kind', default='bernoulli', choices=['bernoulli', 'uniform', 'gaussian'])
    _p.add_argument('--warmup_epochs', type=int, default=2,
                    help='epochs of random-noise-only training before the friendly noise is computed '
                         '(paper: 5 of 80)')
    _p.add_argument('--random_after_warmup', action='store_true',
                    help='keep the warm-up plain: random noise switches on together with the friendly noise '
                         '(the paper has it on from the first epoch)')
    _p.add_argument('--friendly_epochs', type=int, default=30, help='optimization steps for the friendly noise')
    _p.add_argument('--friendly_lr', type=float, default=100.)
    _p.add_argument('--friendly_lambda', type=float, default=1., help='weight of the "make eta large" term')
    _p.add_argument('--friendly_batch', type=int, default=128)
    _p.add_argument('--out_dir', default=os.environ.get('LAYER2_OUT_DIR', os.path.join(HERE, 'results', 'runs')),
                    help='where acc_mat_*.npy and summary_layer2.tsv are written')
    _dargs, _rest = _p.parse_known_args()
    sys.argv = [sys.argv[0]] + _rest
    # <<< LAYER2

    args = get_args()

    # >>> LAYER2
    for _k, _v in vars(_dargs).items():
        setattr(args, _k, _v)
    if args.friends and not 0 <= args.warmup_epochs < args.nepochs:
        sys.exit(f'--warmup_epochs must be in [0, {args.nepochs - 1}] (nepochs={args.nepochs})')
    if args.friends and args.approach not in ('ewc', 'rwalk', 'afec_ewc'):
        sys.exit(f'--friends supports ewc, rwalk and afec_ewc (got {args.approach})')
    # <<< LAYER2

    main(args)
    

## 5. Data and checkpoints (CIFAR-100 plus the authors' victim checkpoints, about 200 MB)

In [ ]:
%cd /content/drive/MyDrive/brainwash/brainwash-repro
!pip -q install huggingface_hub
# checkpoints and CIFAR-100 are stored inside the repo on Drive, so they are only fetched the first time
import os
if not os.path.isdir('repro/artifacts/cifar100_files'):
    !python repro/download_artifacts.py --what cifar100
else:
    print('victim checkpoints already on Drive: skipped')
if not os.path.isdir('data/cifar-100-python'):
    !python repro/prepare_cifar100.py
else:
    print('CIFAR-100 already on Drive: skipped')

## 5b. Build the partially-poisoned noise file

`partial_poison.py` picks a random `POISON_FRAC` of the last task's training samples (seeded by `POISON_SEED`) and zeroes the stage-3 noise
on all the others, saving a derived pkl. Everything below uses that file. The file is cached on Drive, so re-running is instant.
Expected: `poisoned samples = 1000 / 5000` for `POISON_FRAC = 0.2`.

In [ ]:
import importlib.util, pickle
spec = importlib.util.spec_from_file_location('partial_poison', PARTIAL_PY)
pp = importlib.util.module_from_spec(spec); spec.loader.exec_module(pp)

stem = os.path.splitext(os.path.basename(NOISE_PKL))[0]
PARTIAL_PKL = f'{POISON_DIR}/{stem}_ours_p{POISON_FRAC}_ps{POISON_SEED}.pkl'
if POISON_FRAC >= 1.0:
    PARTIAL_PKL = NOISE_PKL                      # full poisoning: use the original file
elif not os.path.isfile(PARTIAL_PKL):
    pp.build_partial_checkpoint(NOISE_PKL, PARTIAL_PKL, POISON_FRAC, POISON_SEED, uniform=False)

# sanity check: count the samples that actually carry noise
ck = pickle.load(open(PARTIAL_PKL, 'rb'))
noise = ck['latest_noise']
N = len(noise)
n_poisoned = int((noise.flatten(1).abs().sum(1) > 0).sum())
print(f'poisoned samples = {n_poisoned} / {len(noise)}  ({n_poisoned/len(noise):.1%} of the last task)')
print('derived noise file:', PARTIAL_PKL)
del ck, noise
assert abs(n_poisoned / N - POISON_FRAC) < 0.01 or POISON_FRAC >= 1.0, 'mask did not apply as expected'

## 5c. (Optional) Check that the partial step changes nothing it shouldn't

1. **Identity at 1.0:** force the wrapper to run with `POISON_FRAC = 1.0`; the derived file must hold exactly the same noise as the original.
2. **Exactness at your fraction:** in the derived file, poisoned samples must carry exactly the original noise and all others exactly zero.
3. **End to end (cell 5d):** run the poisoned/no-defence arm on the forced-1.0 file; its BWT must equal your earlier full-poison arm B.

In [ ]:
import numpy as np, torch, pickle
orig = pickle.load(open(NOISE_PKL, 'rb'))

# 1) identity at 1.0 (forces the wrapper path, which the normal flow skips at 1.0)
CHK_PKL = f'{POISON_DIR}/_check_p1.0.pkl'
pp.build_partial_checkpoint(NOISE_PKL, CHK_PKL, 1.0, POISON_SEED, uniform=False)
der = pickle.load(open(CHK_PKL, 'rb'))
assert torch.equal(orig['latest_noise'], der['latest_noise']), 'latest_noise changed at frac=1.0'
if 'noise' in orig and torch.is_tensor(orig['noise']):
    assert torch.equal(orig['noise'], der['noise']), 'noise changed at frac=1.0'
assert np.array_equal(np.asarray(orig['rnd_idx_train']), np.asarray(der['rnd_idx_train'])), 'sample order changed'
assert set(der) - set(orig) == {'poison_frac', 'poison_seed', 'poison_n'} and set(orig) <= set(der), 'unexpected keys'
for k in ('delta', 'attacked_task', 'target_task'):
    if k in orig: assert orig[k] == der[k], k
print('1) OK: at POISON_FRAC=1.0 the derived file is identical to the original (only poison_* keys added)')

# 2) exactness at the chosen fraction
if POISON_FRAC < 1.0:
    part = pickle.load(open(PARTIAL_PKL, 'rb'))
    nz = part['latest_noise'].flatten(1).abs().sum(1) > 0
    assert torch.equal(part['latest_noise'][nz], orig['latest_noise'][nz]), 'poisoned samples were altered'
    assert bool((part['latest_noise'][~nz] == 0).all()), 'clean samples carry noise'
    print(f'2) OK: {int(nz.sum())} samples carry exactly the original noise, {int((~nz).sum())} are exactly zero')
    del part
del orig, der

### 5d. End to end: arm B on the forced-1.0 file vs your earlier full-poison arm B

This runs the poisoned/no-defence arm (about 5 minutes) on `_check_p1.0.pkl`. Compare its `After BWT` / `Last task acc` with the log of your earlier arm B (`B_poison_s0.log`, from your 100% runs, same `NOISE_PKL`, `SEED`, `LAMB`). With seeds and `cudnn.deterministic` they should be identical, or differ only in the last digits.

In [ ]:
OLD_B_LOG = '/content/drive/MyDrive/brainwash/layer2_runs/B_poison_s0.log'   # <- your earlier full-poison arm B log (edit)
CHECK_DIR = f'{RESULTS_DIR}/_check_p1.0'
os.makedirs(CHECK_DIR, exist_ok=True)
check_log = f'{CHECK_DIR}/B_poison_forced1.0_s{SEED}.log'
!cd {REPO_DIR} && python layer2_friends/friends_layer2.py --experiment split_cifar100 --approach ewc --lasttask 9 --tasknum 10 \
    --nepochs 20 --batch-size 16 --lr 0.01 --clip 100. --lamb {LAMB if 'LAMB' in globals() else 500000} --seed {SEED} \
    --checkpoint "{CHK_PKL}" --init_acc --out_dir "{CHECK_DIR}" --addnoise 2>&1 | grep -v Warning > "{check_log}"
def after(path):
    return [l.strip() for l in open(path) if l.startswith('After BWT')]
print('forced-1.0 :', after(check_log))
print('earlier B  :', after(OLD_B_LOG) if os.path.isfile(OLD_B_LOG) else f'(not found: {OLD_B_LOG})')

## 6. Run the six arms

FRIENDS settings are the paper's: ζ = μ = 16/255, Bernoulli random noise, friendly noise from 30 steps (lr 100, λ = 1),
computed after **2 warm-up epochs** of the 20 (the paper uses 5 of 80). Change `WARMUP` to try another length.

In [ ]:
WARMUP = 2
LAMB = 500000          # EWC on CIFAR-100, as in repro/sweep/configs.tsv

BASE = (f'python layer2_friends/friends_layer2.py --experiment split_cifar100 --approach ewc --lasttask 9 --tasknum 10 '
        f'--nepochs 20 --batch-size 16 --lr 0.01 --clip 100. --lamb {LAMB} --seed {SEED} '
        f'--checkpoint "{PARTIAL_PKL}" --init_acc --out_dir "{RESULTS_DIR}"')
ARMS = [
    ('A_clean',          ''),
    ('B_poison',         '--addnoise'),
    ('F_poison_friends', f'--addnoise --friends --warmup_epochs {WARMUP}'),
    ('Fclean_friends',   f'--friends --warmup_epochs {WARMUP}'),
    ('Ffr_friendly_only', f'--addnoise --friends --components friendly_only --zeta 32 --warmup_epochs {WARMUP}'),
    ('Frn_random_only',   '--addnoise --friends --components random_only --mu_rand 32'),
]
import subprocess, time
for name, flags in ARMS:
    log = f'{RESULTS_DIR}/{name}_s{SEED}.log'
    if os.path.exists(log) and 'appended summary row' in open(log).read():
        print(f'[skip] {name}: already done'); continue
    t0 = time.time()
    print(f'[run ] {name}  ...', flush=True)
    subprocess.run(f'cd /content/drive/MyDrive/brainwash/brainwash-repro && {BASE} {flags} 2>&1 | grep -v Warning > "{log}"', shell=True)
    done = 'appended summary row' in open(log).read()
    print(f'[{"done" if done else "FAIL"}] {name}  ({time.time()-t0:.0f}s)  log: {log}', flush=True)
    if not done:
        print(''.join(open(log).readlines()[-15:]))
        break

## 7. Results

BWT and accuracies are in %, as in the BrainWash paper (more negative BWT = more forgetting; "last-task acc" is the paper's Acc).
`old_acc_at_eta` is the average old-task accuracy at the moment the friendly noise was computed (after the warm-up),
i.e. how much was already forgotten before the defense could act.

In [ ]:
import pandas as pd
df = pd.read_csv(f'{RESULTS_DIR}/summary_layer2.tsv', sep='\t')
df = df[df.seed == SEED]
df['poison_frac'] = df.noise_pkl.str.extract(r'_p([0-9.]+?)_ps')[0].fillna('1.0')   # fraction of the last task poisoned
df['eps'] = df.noise_pkl.str.extract(r'eps([0-9]+\.?[0-9]*)')[0]   # the attack's epsilon, from the noise file name
df = df.drop_duplicates(['eps', 'method'], keep='last').sort_values('eps')
for c in ['bwt_after', 'avg_acc_after', 'last_task_acc']:
    df[c] = (100 * df[c]).round(1)
pd.set_option('display.width', 250)
df[['eps', 'poison_frac', 'method', 'bwt_after', 'avg_acc_after', 'last_task_acc', 'old_acc_at_eta', 'eta_mean_over_zeta', 'eta_frac_at_bound']]

## 8. Send back

Paste the table (and, if anything shows FAIL, the last lines of that arm's `.log`) into the chat.
The arm logs also contain a line starting `[LAYER2] friendly noise computed ...`, which I'll want to see for arm F.